# 02 - Data Cleaning and Preparation

## Objective

The objective of this notebook is to transform the raw Global Superstore dataset into a clean and analysis-ready dataset.

This cleaned dataset will be reused for:

- Exploratory Data Analysis
- SQL analysis
- Excel dashboard
- Power BI dashboard
- Final business report

The goal is not to change the meaning of the data, but to make it easier, safer, and more consistent to analyze.

## Cleaning Strategy

Based on the Data Understanding phase, the following cleaning actions will be performed:

1. Load the raw dataset.
2. Create a working copy to preserve the original data.
3. Rename columns using clear and consistent names.
4. Convert date columns to datetime format.
5. Remove non-useful technical columns.
6. Clean text values.
7. Create new business variables.
8. Run final data quality checks.
9. Export the cleaned dataset.

No rows will be removed automatically during this phase.

## 0. Notebook Setup

This section imports the required Python libraries and sets display options to make tables easier to read.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## 1. Load the Raw Dataset

The raw dataset is loaded from the `data/raw` folder.

A flexible file path is used so the notebook can run:

- locally from the `notebooks` folder;
- locally from the repository root;
- in Google Colab with `superstore.csv` uploaded directly.

In [ ]:
possible_paths = [
    Path("../data/raw/superstore.csv"),  # when notebook is inside /notebooks
    Path("data/raw/superstore.csv"),     # when notebook is executed from repository root
    Path("superstore.csv")               # when using Google Colab or a simple local folder
]

DATA_PATH = None

for path in possible_paths:
    if path.exists():
        DATA_PATH = path
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "The raw dataset was not found. Please check that superstore.csv exists in data/raw/ or in the current folder."
    )

df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset loaded from: {DATA_PATH}")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")

display(df_raw.head())

## 2. Create a Working Copy

The original dataset should never be modified directly.

A working copy is created so that all cleaning transformations are applied safely.

In [ ]:
df_clean = df_raw.copy()

print("Working copy created successfully.")
print(f"Working copy rows: {df_clean.shape[0]:,}")
print(f"Working copy columns: {df_clean.shape[1]}")

## 3. Rename Columns

The original column names contain dots, uppercase letters, and one non-English column name.

For better readability and SQL compatibility, all column names will be converted to `snake_case`.

Example:

`Order.Date` becomes `order_date`.

In [ ]:
rename_columns = {
    "Category": "category",
    "City": "city",
    "Country": "country",
    "Customer.ID": "customer_id",
    "Customer.Name": "customer_name",
    "Discount": "discount",
    "Market": "market",
    "记录数": "record_count",
    "Order.Date": "order_date",
    "Order.ID": "order_id",
    "Order.Priority": "order_priority",
    "Product.ID": "product_id",
    "Product.Name": "product_name",
    "Profit": "profit",
    "Quantity": "quantity",
    "Region": "region",
    "Row.ID": "row_id",
    "Sales": "sales",
    "Segment": "segment",
    "Ship.Date": "ship_date",
    "Ship.Mode": "ship_mode",
    "Shipping.Cost": "shipping_cost",
    "State": "state",
    "Sub.Category": "sub_category",
    "Year": "order_year",
    "Market2": "market_group",
    "weeknum": "order_week"
}

df_clean = df_clean.rename(columns=rename_columns)

column_overview_after_renaming = pd.DataFrame({
    "Original Column": df_raw.columns,
    "Clean Column": df_clean.columns
})

display(column_overview_after_renaming)

### Observation

Column names are now easier to read and easier to use in Python, SQL, Excel, and Power BI.

## 4. Convert Date Columns

The columns `order_date` and `ship_date` are currently stored as text.

They need to be converted into date format to allow time-based analysis such as:

- monthly sales trends;
- yearly profit trends;
- shipping delay calculation;
- seasonal analysis.

In [ ]:
df_clean["order_date"] = pd.to_datetime(df_clean["order_date"], errors="coerce")
df_clean["ship_date"] = pd.to_datetime(df_clean["ship_date"], errors="coerce")

date_quality_check = pd.DataFrame({
    "Column": ["order_date", "ship_date"],
    "Missing values after conversion": [
        df_clean["order_date"].isna().sum(),
        df_clean["ship_date"].isna().sum()
    ],
    "Data type": [
        str(df_clean["order_date"].dtype),
        str(df_clean["ship_date"].dtype)
    ],
    "Minimum date": [
        df_clean["order_date"].min(),
        df_clean["ship_date"].min()
    ],
    "Maximum date": [
        df_clean["order_date"].max(),
        df_clean["ship_date"].max()
    ]
})

display(date_quality_check)

## 5. Validate Existing Time Columns

The raw dataset already contains `Year` and `weeknum`, renamed as `order_year` and `order_week`.

This step checks whether `order_year` is consistent with the year extracted from `order_date`.

In [ ]:
df_clean["order_year_from_date"] = df_clean["order_date"].dt.year

year_mismatch_count = (df_clean["order_year"] != df_clean["order_year_from_date"]).sum()

time_validation_summary = pd.DataFrame({
    "Check": [
        "Rows where order_year does not match order_date year",
        "Unique order_year values",
        "Unique order_week values"
    ],
    "Result": [
        year_mismatch_count,
        sorted(df_clean["order_year"].dropna().unique()),
        df_clean["order_week"].nunique()
    ]
})

display(time_validation_summary)

### Decision

The helper column `order_year_from_date` was created only to validate the original `order_year` column.

It will not be kept in the final cleaned dataset because `order_year` already exists.

In [ ]:
df_clean = df_clean.drop(columns=["order_year_from_date"])

print("Temporary validation column removed.")

## 6. Remove Non-Useful Technical Columns

The column `record_count` appears to be a technical counter column.

Since it does not provide business value for this analysis, it will be removed from the cleaned dataset.

In [ ]:
if "record_count" in df_clean.columns:
    print("Unique values in record_count:")
    display(df_clean["record_count"].value_counts().to_frame("count"))

    df_clean = df_clean.drop(columns=["record_count"])
    print("record_count column removed.")
else:
    print("record_count column not found.")

## 7. Clean Text Columns

Text columns may contain unnecessary spaces.

This step removes leading and trailing spaces from all text columns.

In [ ]:
text_columns = df_clean.select_dtypes(include="object").columns

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

print(f"Text cleaning applied to {len(text_columns)} text columns.")
display(pd.DataFrame({"Text Columns Cleaned": text_columns}))

## 8. Create Business Variables

New variables will be created to make the dataset more useful for business analysis.

These variables will help analyze profitability, shipping efficiency, discounts, order size, and time-based trends.

### 8.1 Create Time Variables

These variables will make it easier to analyze sales and profit by month, quarter, and year-month.

In [ ]:
df_clean["order_month"] = df_clean["order_date"].dt.month
df_clean["order_month_name"] = df_clean["order_date"].dt.month_name()
df_clean["order_quarter"] = "Q" + df_clean["order_date"].dt.quarter.astype(str)
df_clean["order_year_month"] = df_clean["order_date"].dt.to_period("M").astype(str)

display(df_clean[[
    "order_date",
    "order_year",
    "order_month",
    "order_month_name",
    "order_quarter",
    "order_year_month",
    "order_week"
]].head())

### 8.2 Create Shipping Delay

`shipping_delay_days` measures the number of days between the order date and the shipping date.

This variable will be useful for analyzing shipping performance and operational efficiency.

In [ ]:
df_clean["shipping_delay_days"] = (
    df_clean["ship_date"] - df_clean["order_date"]
).dt.days

shipping_delay_summary = df_clean["shipping_delay_days"].describe().to_frame("shipping_delay_days")

display(shipping_delay_summary)

### 8.3 Create Profit Margin

`profit_margin` shows how much profit is generated for each unit of sales.

A positive margin means the transaction is profitable.

A negative margin means the transaction generated a loss.

Rows with zero sales are not deleted automatically. Their profit margin is set as missing because division by zero is not meaningful.

In [ ]:
df_clean["profit_margin"] = np.where(
    df_clean["sales"] != 0,
    df_clean["profit"] / df_clean["sales"],
    np.nan
)

display(df_clean[["sales", "profit", "profit_margin"]].head())

### 8.4 Create Profit Status

`profit_status` classifies each transaction as:

- `Profitable`
- `Loss-making`
- `Break-even`

This will make it easier to compare profitable and unprofitable transactions.

In [ ]:
df_clean["profit_status"] = np.select(
    [
        df_clean["profit"] > 0,
        df_clean["profit"] < 0,
        df_clean["profit"] == 0
    ],
    [
        "Profitable",
        "Loss-making",
        "Break-even"
    ],
    default="Unknown"
)

display(df_clean["profit_status"].value_counts().to_frame("count"))

### 8.5 Create Discount Band

`discount_band` groups discount values into business-friendly categories.

This will make it easier to analyze whether higher discounts improve or reduce profitability.

In [ ]:
conditions = [
    df_clean["discount"] == 0,
    (df_clean["discount"] > 0) & (df_clean["discount"] <= 0.10),
    (df_clean["discount"] > 0.10) & (df_clean["discount"] <= 0.20),
    (df_clean["discount"] > 0.20) & (df_clean["discount"] <= 0.30),
    (df_clean["discount"] > 0.30) & (df_clean["discount"] <= 0.50),
    df_clean["discount"] > 0.50
]

choices = [
    "No discount",
    "Low discount",
    "Moderate discount",
    "High discount",
    "Very high discount",
    "Extreme discount"
]

df_clean["discount_band"] = np.select(conditions, choices, default="Unknown")

discount_band_order = [
    "No discount",
    "Low discount",
    "Moderate discount",
    "High discount",
    "Very high discount",
    "Extreme discount",
    "Unknown"
]

display(
    df_clean["discount_band"]
    .value_counts()
    .reindex(discount_band_order)
    .dropna()
    .to_frame("count")
)

### 8.6 Create Shipping Cost Ratio

`shipping_cost_ratio` measures shipping cost relative to sales.

This helps identify transactions where shipping costs represent a large share of revenue.

In [ ]:
df_clean["shipping_cost_ratio"] = np.where(
    df_clean["sales"] != 0,
    df_clean["shipping_cost"] / df_clean["sales"],
    np.nan
)

display(df_clean[["sales", "shipping_cost", "shipping_cost_ratio"]].head())

### 8.7 Create Order Size

`order_size` groups order lines based on quantity.

This variable can help compare small and large order lines.

In [ ]:
df_clean["order_size"] = pd.cut(
    df_clean["quantity"],
    bins=[0, 2, 5, 10, np.inf],
    labels=["Small", "Medium", "Large", "Very Large"],
    include_lowest=True
)

display(df_clean["order_size"].value_counts().to_frame("count"))

## 9. Review Zero-Sales Rows

Rows with zero sales should not be removed automatically.

They need to be reviewed because they may represent valid records, rounding issues, or data entry problems.

In [ ]:
zero_sales_count = (df_clean["sales"] == 0).sum()

print(f"Number of rows with zero sales: {zero_sales_count}")

if zero_sales_count > 0:
    display(df_clean[df_clean["sales"] == 0])
else:
    print("No zero-sales rows were found.")

### Decision

Rows with zero sales will be kept in the dataset for now.

They can be reviewed during the Exploratory Data Analysis phase, but they should not be removed without a clear business reason.

## 10. Reorder Columns

The cleaned dataset will be reordered to make it easier to read.

Columns are grouped by business area:

1. Order information
2. Customer information
3. Product information
4. Geography
5. Financial metrics
6. Shipping metrics
7. Created business variables

In [ ]:
preferred_column_order = [
    "row_id",
    "order_id",
    "order_date",
    "ship_date",
    "order_year",
    "order_month",
    "order_month_name",
    "order_quarter",
    "order_year_month",
    "order_week",
    "order_priority",
    "customer_id",
    "customer_name",
    "segment",
    "product_id",
    "product_name",
    "category",
    "sub_category",
    "country",
    "state",
    "city",
    "region",
    "market",
    "market_group",
    "sales",
    "quantity",
    "discount",
    "profit",
    "shipping_cost",
    "ship_mode",
    "shipping_delay_days",
    "profit_margin",
    "profit_status",
    "discount_band",
    "shipping_cost_ratio",
    "order_size"
]

existing_preferred_columns = [col for col in preferred_column_order if col in df_clean.columns]
remaining_columns = [col for col in df_clean.columns if col not in existing_preferred_columns]

df_clean = df_clean[existing_preferred_columns + remaining_columns]

display(pd.DataFrame({"Final Column Order": df_clean.columns}))

## 11. Final Data Quality Checks

After cleaning, the dataset must be checked again to make sure that no major issue was introduced during the transformation process.

In [ ]:
final_quality_checks = pd.DataFrame({
    "Check": [
        "Original row count",
        "Cleaned row count",
        "Total columns",
        "Missing values",
        "Duplicated rows",
        "Duplicated row_id",
        "Rows with ship_date before order_date",
        "Negative sales",
        "Negative quantity",
        "Negative shipping cost",
        "Discount below 0",
        "Discount above 1",
        "Rows with zero sales",
        "Rows with missing profit_margin",
        "Columns still containing dots"
    ],
    "Result": [
        df_raw.shape[0],
        df_clean.shape[0],
        df_clean.shape[1],
        df_clean.isnull().sum().sum(),
        df_clean.duplicated().sum(),
        df_clean["row_id"].duplicated().sum(),
        (df_clean["ship_date"] < df_clean["order_date"]).sum(),
        (df_clean["sales"] < 0).sum(),
        (df_clean["quantity"] < 0).sum(),
        (df_clean["shipping_cost"] < 0).sum(),
        (df_clean["discount"] < 0).sum(),
        (df_clean["discount"] > 1).sum(),
        (df_clean["sales"] == 0).sum(),
        df_clean["profit_margin"].isna().sum(),
        len([col for col in df_clean.columns if "." in col])
    ]
})

display(final_quality_checks)

### Interpretation

The row count should remain unchanged because no rows are intentionally removed during this cleaning phase.

Some missing values may appear in newly created ratio columns if sales are equal to zero. This is expected because division by zero is not meaningful.

## 12. Cleaned Dataset Overview

This section displays the final structure of the cleaned dataset.

In [ ]:
print(f"Final rows: {df_clean.shape[0]:,}")
print(f"Final columns: {df_clean.shape[1]}")

display(df_clean.head())

In [ ]:
df_clean.info()

In [ ]:
cleaned_column_overview = pd.DataFrame({
    "Column": df_clean.columns,
    "Data Type": df_clean.dtypes.astype(str).values,
    "Missing Values": df_clean.isnull().sum().values,
    "Unique Values": df_clean.nunique(dropna=False).values
})

display(cleaned_column_overview)

## 13. Cleaning Log

The table below summarizes the main cleaning actions applied to the dataset.

In [ ]:
cleaning_log = pd.DataFrame({
    "Step": [
        "Created working copy",
        "Renamed columns",
        "Converted date columns",
        "Validated existing time columns",
        "Removed record_count",
        "Cleaned text columns",
        "Created time variables",
        "Created shipping_delay_days",
        "Created profit_margin",
        "Created profit_status",
        "Created discount_band",
        "Created shipping_cost_ratio",
        "Created order_size",
        "Reviewed zero-sales rows",
        "Reordered columns"
    ],
    "Reason": [
        "Preserve the raw dataset",
        "Improve readability and SQL compatibility",
        "Enable time-based analysis",
        "Check consistency between existing year column and order date",
        "Remove non-useful technical counter column",
        "Remove unnecessary leading/trailing spaces",
        "Support monthly, quarterly, and yearly analysis",
        "Measure shipping efficiency",
        "Measure profitability relative to sales",
        "Classify transactions by profitability",
        "Analyze discounts using business-friendly groups",
        "Measure shipping cost relative to sales",
        "Group order lines by quantity size",
        "Avoid deleting records without business justification",
        "Improve readability of the cleaned dataset"
    ]
})

display(cleaning_log)

## 14. Export the Cleaned Dataset

The cleaned dataset will be saved in a `data/cleaned` folder when possible.

This file will be used in the next phases of the project.

In [ ]:
# Choose the best output folder depending on the execution environment.
if DATA_PATH.parent.name == "raw" and DATA_PATH.parent.parent.name == "data":
    OUTPUT_DIR = DATA_PATH.parent.parent / "cleaned"
elif Path("../data").exists():
    OUTPUT_DIR = Path("../data/cleaned")
elif Path("data").exists():
    OUTPUT_DIR = Path("data/cleaned")
else:
    OUTPUT_DIR = Path("cleaned")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cleaned_file_path = OUTPUT_DIR / "cleaned_superstore.csv"
cleaning_log_path = OUTPUT_DIR / "cleaning_log.csv"
cleaned_column_overview_path = OUTPUT_DIR / "cleaned_column_overview.csv"

df_clean.to_csv(cleaned_file_path, index=False)
cleaning_log.to_csv(cleaning_log_path, index=False)
cleaned_column_overview.to_csv(cleaned_column_overview_path, index=False)

print(f"Cleaned dataset saved to: {cleaned_file_path}")
print(f"Cleaning log saved to: {cleaning_log_path}")
print(f"Column overview saved to: {cleaned_column_overview_path}")

## 15. Conclusion

The Global Superstore dataset has been cleaned and prepared for analysis.

The main cleaning actions included:

- Renaming columns using clear and consistent names.
- Converting date columns to datetime format.
- Removing the non-useful `record_count` column.
- Cleaning text values.
- Creating new business variables such as:
  - `profit_margin`
  - `shipping_delay_days`
  - `profit_status`
  - `discount_band`
  - `shipping_cost_ratio`
  - `order_size`
  - `order_month`
  - `order_quarter`
  - `order_year_month`

No rows were removed during this cleaning phase.

The cleaned dataset is now ready for Exploratory Data Analysis, SQL analysis, Excel, and Power BI.